# RNA-seq → `expr.h5` on Illinois Computes Research Notebooks (ICRN)

Same job as `expression_gpu.ipynb`, for UIUC's free JupyterHub instead of Colab.

**Start:** open https://jupyter.ncsa.illinois.edu, log in with your NetID, pick the server option
with an **A100 GPU slice**, and start it. Upload this notebook with the file browser's upload
button, open it, and run one cell at a time, comparing with **Expected**.

Everything lives in your ICRN home folder (100 GB, kept between sessions), so a disconnect loses
nothing: run cell 1 again, then continue where you were. The model runs in its own Python
environment (`~/venvs/bulkrnabert`), so packages preinstalled on ICRN can't clash with it.

**1. Setup** (run first in every session). Downloads the three code repos into your home folder.
Expected: no `fatal` lines, the latest tools commit, BulkRNABert at `c85162f`, and a GPU line
from `nvidia-smi` (an A100 with a `MIG` device).

In [ ]:
import os, shlex, subprocess
HOME = os.path.expanduser('~')
BASE = f'{HOME}/tcga10'                                  # all outputs (persistent)
TOOLS = f'{HOME}/tcga-multimodal-embeddings'
AUTH = f'{HOME}/multimodal-cancer-modeling'
BRB = f'{HOME}/BulkRNABert'
VENV = f'{HOME}/venvs/bulkrnabert'
PY = f'{VENV}/bin/python'
CASES = f'{BASE}/split_cases.csv'
EMBED = (f'{PY} {TOOLS}/scripts/embed_expr_bulkrnabert.py --expr-dir {BASE}/expr_raw '
         f'--gene-list {AUTH}/data/bulkrnabert_gene_list.txt '
         f'--weights-folder {BRB}/checkpoints --output-h5 {BASE}/expr.h5')
os.makedirs(BASE, exist_ok=True)
for d in (TOOLS, AUTH, BRB):  # ICRN home folders trip git's 'dubious ownership' check
    !git config --global --get-all safe.directory | grep -qxF {d} || git config --global --add safe.directory {d}
![ -d {TOOLS} ] || git clone -q https://github.com/YaseenM2/tcga-multimodal-embeddings.git {TOOLS}
!git -C {TOOLS} pull -q
![ -d {AUTH} ] || git clone -q --depth 1 https://github.com/StevenSong/multimodal-cancer-modeling.git {AUTH}
![ -d {BRB} ] || git clone -q https://github.com/StevenSong/BulkRNABert.git {BRB}
!git -C {BRB} checkout -q c85162ff18913313d0bdea4c3c1bc6d00f69b193
!git -C {TOOLS} log --oneline -1 && git -C {BRB} log --oneline -1
%cd {TOOLS}
!nvidia-smi -L

**2. Private Python environment** (first session only; about 5 minutes). Expected: an
`installer:` line, `73 passed`, then `0.5.3` and a list with a `CudaDevice`. If it's interrupted,
just run it again: a half-built environment is detected and rebuilt.

In [ ]:
# Rebuild the environment if its pip is broken (e.g. after an interrupted first run).
pip_ok = lambda: os.path.exists(PY) and subprocess.run(
    [PY, '-c', 'import pip._internal.commands.install'], capture_output=True).returncode == 0
if not pip_ok():
    !rm -rf {VENV} && python3 -m venv {VENV}
if pip_ok():
    INSTALL = f'{PY} -m pip install -q'
else:  # fall back to uv, which doesn't need the environment's own pip
    !python3 -m pip install -q --user uv
    INSTALL = f'{HOME}/.local/bin/uv pip install -q --python {PY}'
print('installer:', INSTALL.split()[0])
!{INSTALL} -r requirements.txt -r requirements-expression.txt
!{INSTALL} --no-deps -e {BRB}
!{PY} -m pytest -q -p no:cacheprovider tests/ 2>&1 | tail -1
!{PY} -c "import jax; print(jax.__version__, jax.devices())"

**3. Patient list, then each patient's RNA-seq files** (public GDC API, no login).
Expected: `TOTAL 873 306 567 ...`, then JSON with `kept_cases: 873`, `kept_files: 873` (or close),
`study_cases_without_file: 0`.

In [ ]:
!{PY} scripts/select_cohort.py --clinical {AUTH}/data/clinical.csv --splits {AUTH}/results/split_cases.csv --out-dir {BASE} | tail -3
!{PY} scripts/prepare_expression_manifest.py --expr-manifest {AUTH}/data/expr-manifest.txt --splits {CASES} --out-map {BASE}/expr_files_tcga10.csv --out-manifest {BASE}/expr-manifest-tcga10.txt

**4. Download** (about 3.5 GB, MD5-checked; rerun to resume).
Expected last line: `ALL EXPRESSION FILES PRESENT AND VERIFIED!`.

In [ ]:
!{PY} scripts/download_expression.py --map {BASE}/expr_files_tcga10.csv --out-dir {BASE}/expr_raw

**5. Pilot: 5 files.** Expected: `embedded 1 files (... s/file)` lines and `new: 5`.
Use the `s/file` value to estimate how long step 6 takes.

In [ ]:
!{EMBED} --max-new 5

**6. Everything else,** in batches of 100 (each batch prints a progress line). After a disconnect:
cell 1, then this cell. Expected final line: `EXPRESSION EMBEDDING COMPLETE!`.

In [ ]:
while True:
    run = subprocess.run(shlex.split(EMBED) + ['--max-new', '100'], capture_output=True, text=True)
    if run.returncode != 0:
        print(run.stdout[-1500:], run.stderr[-3000:]); break
    print(' | '.join(l for l in run.stdout.splitlines() if l.split(':')[0] in ('new', 'skipped', 'remaining')))
    if 'EMBEDDING COMPLETE' in run.stdout:
        print('EXPRESSION EMBEDDING COMPLETE!'); break

**7. Check and export.** Expected: `EXPRESSION AUDIT PASSED!` and `"cases_written": 873`.
`expr.h5` and `expr.parquet` are in `~/tcga10`. To copy one to your computer: in the file browser
on the left, open `tcga10`, right-click the file, and choose *Download*.

In [ ]:
check = subprocess.run(shlex.split(EMBED) + ['--check-only'], capture_output=True, text=True)
print(check.stdout.strip())
if check.returncode == 0:
    !{PY} scripts/export_parquet.py --modality expr --h5 {BASE}/expr.h5 --cases {CASES} --output {BASE}/expr.parquet
else:
    print('Not finished yet: run cell 6 again (it resumes), then this cell.')